<a href="https://colab.research.google.com/github/HLZHarry/AlphaLink-Muti-Modal-RAG/blob/main/Step_4_The_Vector_Brain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pathlib import Path
import os
from google.colab import drive

In [5]:
# 1. CLEANUP & MOUNT
# Remove any local 'ghost' folders before mounting
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    print("🧹 Removing local 'ghost' drive folder...")
    !rm -rf /content/drive

# Mount the real Drive
print("🔄 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

🔄 Mounting Google Drive...
Mounted at /content/drive


In [6]:
# 2. Install Necessary Tools
!pip install -q langchain_text_splitters langchain_community chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 

In [7]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [8]:
# 3. PATHS
BASE_DIR = Path("/content/drive/MyDrive/AlphaLink-RAG/data")
MD_DIR = BASE_DIR / "markdown_filings"
DB_DIR = BASE_DIR / "chroma_db"

In [9]:
# 4. CHUNKING STRATEGY
# We split by headers (#) to keep financial sections (like 'Risk Factors') together
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

In [ ]:
# 5. Load The Embedding Model
print ("Loading Embedding Model to GPU...")
embeddings = HuggingFaceEmbeddings(
    model_name = "all_MiniLM-L6-v2",
    model_kwargs = {'device': 'cuda'} # Use T4 GPU
)